In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.signal import savgol_filter, butter, filtfilt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D
%load_ext rpy2.ipython

Import data, set plotting params

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,

    "svg.fonttype": "none",   
    "pdf.fonttype": 42,       
    "ps.fonttype": 42,      
    "text.usetex": False,    
})

In [ ]:
def lowpass(x, fs, cutoff_hz=0.3, order=3):
    b, a = butter(order, cutoff_hz / (fs / 2.0), btype="low")
    return filtfilt(b, a, np.asarray(x, float))

In [ ]:
df = pd.read_csv(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\ResidualTrialDF_Reward.csv')
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)

successdf = df.groupby(['Animal', 'Session', 'Trial'], observed=True).first().reset_index()[['Animal', 'Session', 'Trial', 'Time to Reward', 'Time to Monster']]
successdf['Rewarded'] = np.where(successdf['Time to Reward'].notna(), 1, 0)
successdf = successdf.groupby(['Animal', 'Session'], observed=True).agg("mean").reset_index()
goodanimals  = successdf[(successdf['Session'] == 'Reward') & (successdf['Rewarded'] >= 0.8)]['Animal'].to_list()

df = df[df['Animal'].isin(goodanimals)]
df['Site'] = df['Region']
df['Region'] = df['Site'].str[:2]

df = (df.sort_values(["Trial","Animal","Site","Time to Shelter"],
                     kind="mergesort", na_position="last")
        .reset_index(drop=True))

df['Outcome'] = np.where(df['Time to Reward'].notna(), 'Rewarded', 'Unrewarded')
df['MonsterCharge'] = np.where(df['Time to Monster'].notna(), 'Charge', 'NoCharge')
df['Outcome_Monster'] = df['Outcome']+df['MonsterCharge']

df['v_reward_radial'] = df['v_reward_radial']*-1

df = df[~((df['Region']=='TS')&(df['Animal']=='Wanda'))].copy()
df = df[~((df['Site']=='TSR')&(df['Animal']=='Chester'))].copy()


In [ ]:
parts = []
for g_names, g, in df.groupby(['Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    outcome, session = g_names[0], g_names[1]
    #sampling + window
    fs  = float(1.0 / g['dt'].iloc[0])                 
    win = int(round(fs * 1.5)); win += 1 - (win % 2)  
    win = max(win, 3)

    if len(g) < win:
        continue

    # smooth position + derivative
    g['filtered_x']   = savgol_filter(g['X'].to_numpy(), win, 1, mode='interp')
    g['filtered_y']   = savgol_filter(g['Y'].to_numpy(), win, 1, mode='interp')
    dy                = savgol_filter(g['X'].to_numpy(), win, 1, deriv=1, delta=1/fs, mode='interp')
    g['d_filtered_x'] = lowpass(dy, fs, cutoff_hz=0.3)   
    g.loc[abs(g['d_filtered_x']) <2, 'd_filtered_x' ] = 0 

    #base labels (separate steps for clarity)
    g['approach'] = np.nan
    g['approach'] = g['approach'].astype('object')
    g.loc[g['filtered_x'] < 30,  'approach'] = 'Distal Approach' 
    g.loc[g['filtered_x'] >= 30, 'approach'] = 'Proximal Approach' 
    g.loc[g['filtered_x'] >= 43, 'approach'] = 'Past Spout'
    g.loc[g['d_filtered_x'] < 0, 'approach'] = 'Retreat' 

    #reward -> first retreat becomes 40
    rewardix = g['Time to Reward'].abs().idxmin()
    monsterix = (g['Time to Monster'].abs()).idxmin()
    consumptionix = np.nanmin([g.loc[(g.index > rewardix) & (g['approach'] == 'Retreat')].index.min(), g.loc[(g.index > rewardix) & (g['approach'] == 'Past Spout')].index.min()])
    g.loc[rewardix:consumptionix, 'approach'] = 'Licking'

    g.loc[(g['filtered_x'] <= 0), 'approach'] = np.nan

    lab  = g['approach'].to_numpy() 
    prev = np.concatenate([lab[:1], lab[:-1]]) 

    transitions = (prev == 'Retreat') & (lab != 'Retreat')
    g['approach_counter'] = np.cumsum(transitions.astype(int))

    parts.append(g)
    
df = pd.concat(parts, axis=0).reset_index(drop=True)

df = df[df['approach'].notna()]

parts = []
for gname, g in df.groupby(['approach_counter', 'Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    rmin, rmax = np.min(g['Time to Reward']), np.max(g['Time to Reward'])
    mmin, mmax = np.min(g['Time to Monster']), np.max(g['Time to Monster'])
    if np.isnan(rmin):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax<0)):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax>0)):
        g['Reward'] = 'Rewarded'
    elif ((rmin>0) & (rmax>0)):
        g['Reward'] = 'PostRewarded'

    if np.isnan(mmin):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax<0)):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax>0)):
        g['Monster'] = 'MonsterCharge'
    elif ((mmin>0) & (mmax>0)):
        g['Monster'] = 'MonsterCharging'

    parts.append(g)
df = pd.concat(parts, axis=0).reset_index(drop=True)

In [ ]:
df["Facing_Spout"] = "Side"
df.loc[(df['heading_relative_to_reward'] >= -20) & (df['heading_relative_to_reward'] <= 20), "Facing_Spout"] = "Front"
df.loc[(df['heading_relative_to_reward'] >= 90) | (df['heading_relative_to_reward'] <= -90), "Facing_Spout"] = "Behind"

df["Distance.to.Spout.fac"] = "Distal"
df.loc[(df['Distance to Spout'] <= 10), "Distance.to.Spout.fac"] = "Proximal"

df.rename(columns={'Distance to Spout': 'Distance.to.Spout'}, inplace=True)

df.loc[df["Site"] == "TSR", "heading_relative_to_reward"] *= -1
df.loc[
    (df["Site"] == "VS") & (~df["Animal"].str.contains("EI")),
    "heading_relative_to_reward"] *= -1

df['ipsi_contra'] = "Contra"
df.loc[(df['heading_relative_to_reward'] <= 0) , "ipsi_contra"] = "Ipsi"

In [ ]:
g = ["Region","Session", "Animal","Trial","Site"]
k = np.where(np.isclose(df.dt, 1/12, atol=1e-9), 12, 20)

num = [c for c in df.select_dtypes("number").columns if c not in g]
obj = [c for c in df.columns if c not in num+g]

df = (df.assign(_k=k, _i=df.groupby(g).cumcount(), bin=lambda d: d._i//d._k)
        .groupby(g+["bin"], as_index=False, observed=True)
        .agg({**{c:"mean" for c in num}, **{c:"first" for c in obj}})
        .drop(columns=["bin"])
        .assign(Z_lag1=lambda d: d.groupby(g)["Z"].shift(1))
     )

df = df[df['X']<=40].copy()

Figure 7A

%%R -i df -o v_mean

library(lme4)
library(emmeans)

df$Trial <- as.factor(df$Trial)
df$approach_counter <- as.factor(df$approach_counter)

m1 <- lmer(Z ~ ipsi_contra * Region *Session  + Z_lag1 + (1 |Animal) + (1 | Trial) + (1|Animal:Site), df)
acf(resid(m1))

v_mean <- emmeans(m1, ~ ipsi_contra | Region*Session)
v_test <- test(v_mean, adjust = "bonferroni")

v_mean_df <- data.frame(v_test)

print(v_mean_df)

v_mean <- emmeans(m1,~  ipsi_contra | Region*Session)
print(as.data.frame(test(v_mean, adjust = "bonferroni")))
print(test(pairs(v_mean), adjust = "bonferroni"))
v_mean <- as.data.frame(v_mean)

anova(m1)

Figure 7B, 7C

In [ ]:
%%R -i df -o v_mean

library(lme4)
library(emmeans)

df$Trial <- as.factor(df$Trial)
df$approach_counter <- as.factor(df$approach_counter)

m1 <- lmer(Z ~ Distance.to.Spout.fac * Facing_Spout * Region *Session  + Z_lag1 + (1 |Animal) + (1 | Trial) + (1|Animal:Site), data = df)
acf(resid(m1))

v_mean <- emmeans(m1, ~ Facing_Spout * Distance.to.Spout.fac | Region | Session)
v_test <- test(v_mean, adjust = "bonferroni")

v_mean_df <- data.frame(v_test)

print(v_mean_df)

v_mean <- emmeans(m1,~  Facing_Spout * Distance.to.Spout.fac | Region | Session)
#print(as.data.frame(test(v_mean, adjust = "bonferroni")))
#print(test(pairs(v_mean), adjust = "bonferroni"))
v_mean <- as.data.frame(v_mean)



In [ ]:
REGIONS      = ["VS", "TS"]
SESSIONS     = ["Reward", "Monster"]

row_fac      = "Session"
col_fac      = "Region"
hue_col      = "Animal"

facing_col   = "Facing_Spout"
dist_fac_col = "Distance.to.Spout.fac"
y_col        = "Z"

DIST_LEVELS   = ["Distal", "Proximal"]
FACING_LEVELS = ["Front", "Side", "Behind"]

POINT_ALPHA = 0.25
POINT_SIZE  = 14
JITTER      = 0.04

MEAN_MS      = 6
MEAN_LW      = 2.0
MEAN_CAPSIZE = 4

MARKER = "o" 
YLIM   = (-0.1, 0.5)

REGION_COLORS = {"VS": "#1f77b4", "TS": "#E68600"}

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 9.5,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 150,
})


df_plot = df.copy()
df_plot = df_plot[df_plot[col_fac].isin(REGIONS)].copy()

for c in [row_fac, col_fac, hue_col, facing_col, dist_fac_col]:
    df_plot[c] = df_plot[c].astype(str).str.strip()

df_plot[row_fac]     = pd.Categorical(df_plot[row_fac],     categories=SESSIONS,      ordered=True)
df_plot[col_fac]     = pd.Categorical(df_plot[col_fac],     categories=REGIONS,       ordered=True)
df_plot[dist_fac_col]= pd.Categorical(df_plot[dist_fac_col],categories=DIST_LEVELS,   ordered=True)
df_plot[facing_col]  = pd.Categorical(df_plot[facing_col],  categories=FACING_LEVELS, ordered=True)

animal_means = (
    df_plot
    .groupby([row_fac, col_fac, dist_fac_col, facing_col, hue_col], observed=True, sort=False)
    .agg(meanZ=(y_col, "mean"))
    .reset_index()
)

trend = v_mean.copy()
trend = trend.rename(columns=lambda c: c.strip())

for c in [row_fac, col_fac, facing_col, dist_fac_col]:
    trend[c] = trend[c].astype(str).str.strip()

if "emmean" not in trend.columns:
    raise ValueError(f"Expected 'emmean' in v_mean; found columns: {list(trend.columns)}")

trend = trend[trend[col_fac].isin(REGIONS)].copy()
trend[row_fac]      = pd.Categorical(trend[row_fac],      categories=SESSIONS,      ordered=True)
trend[col_fac]      = pd.Categorical(trend[col_fac],      categories=REGIONS,       ordered=True)
trend[dist_fac_col] = pd.Categorical(trend[dist_fac_col], categories=DIST_LEVELS,   ordered=True)
trend[facing_col]   = pd.Categorical(trend[facing_col],   categories=FACING_LEVELS, ordered=True)

dist_pos = {"Distal": 0.0, "Proximal": 1.0}

span = 0.36
face_offsets = dict(zip(FACING_LEVELS, np.linspace(-span/2, span/2, len(FACING_LEVELS))))

fig, axs = plt.subplots(
    nrows=len(SESSIONS), ncols=len(REGIONS),
    sharex=True, sharey=True,
    figsize=(4.8 * len(REGIONS), 4.6 * len(SESSIONS)),
    constrained_layout=False
)
axs = np.atleast_2d(axs)

rng = np.random.default_rng(1)

for i, sess in enumerate(SESSIONS):
    for j, region in enumerate(REGIONS):
        ax = axs[i, j]
        col = REGION_COLORS[region]

        sub = animal_means[(animal_means[row_fac] == sess) & (animal_means[col_fac] == region)]

        for d in DIST_LEVELS:
            x_base = dist_pos[d]
            sub_d = sub[sub[dist_fac_col] == d]

            for f in FACING_LEVELS:
                sub_df = sub_d[sub_d[facing_col] == f]
                if sub_df.empty:
                    continue

                xs = x_base + face_offsets[f] + rng.uniform(-JITTER, +JITTER, size=len(sub_df))
                ys = sub_df["meanZ"].to_numpy()

                ax.scatter(
                    xs, ys,
                    s=POINT_SIZE, alpha=POINT_ALPHA,
                    color=col, marker=MARKER,
                    edgecolors="none",
                    zorder=2
                )
        t = trend[(trend[row_fac] == sess) & (trend[col_fac] == region)]
        for d in DIST_LEVELS:
            x_base = dist_pos[d]
            t_d = t[t[dist_fac_col] == d]

            for f in FACING_LEVELS:
                one = t_d[t_d[facing_col] == f]
                if one.empty:
                    continue

                m  = float(pd.to_numeric(one["emmean"], errors="coerce").iloc[0])
                se = float(pd.to_numeric(one["SE"], errors="coerce").iloc[0]) if "SE" in one.columns else np.nan
                if not np.isfinite(m):
                    continue

                ax.errorbar(
                    x_base + face_offsets[f], m,
                    yerr=se if np.isfinite(se) else None,
                    fmt=MARKER,
                    color=col,
                    mfc="white", mec=col, ecolor=col,
                    ms=MEAN_MS, lw=MEAN_LW, capsize=MEAN_CAPSIZE,
                    zorder=10, clip_on=False
                )

        ax.axhline(0, linestyle=(0, (4, 3)), linewidth=1.0, color="0.4", alpha=0.75, zorder=1)
        sns.despine(ax=ax, top=True, right=True)

        ax.set_xlim(-0.5, 1.5)
        if YLIM is not None:
            ax.set_ylim(*YLIM)

        if i == 0:
            ax.set_title(region, pad=2)
        if j == 0:
            ax.set_ylabel(sess)

        ax.set_xticks([dist_pos["Distal"], dist_pos["Proximal"]])
        ax.set_xticklabels(["Distal", "Proximal"])
        ax.tick_params(axis="x", labelbottom=True)

fig.supylabel("Mean Z", x=0.02)
fig.supxlabel("Distance bin", y=0.02)

facing_handles = [
    Line2D([0],[0], marker=MARKER, linestyle="None",
           markerfacecolor="white", markeredgecolor="k",
           markersize=6, label=f"{f} ({'left' if face_offsets[f] < 0 else 'center' if face_offsets[f] == 0 else 'right'})")
    for f in FACING_LEVELS
]
fig.legend(handles=facing_handles, loc="center left", bbox_to_anchor=(0.995, 0.5),
           frameon=False, title="Facing (dodge)")

fig.tight_layout(rect=(0.04, 0.04, 0.97, 0.98))
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Zoom_Mean_AngleDist.svg')
plt.show()

Figure 7D

In [ ]:
%%R -i df -o v_slope

library(lme4)
library(emmeans)

df$Trial <- as.factor(df$Trial)
df$approach_counter <- as.factor(df$approach_counter)

m1 <- lmer(Z ~ Distance.to.Spout * Facing_Spout * Region *Session  + Z_lag1 + (1 |Animal) + (1 | Trial) + (1|Animal:Site), data = df)
acf(resid(m1))

v_slope <- emtrends(m1,~  Region | Facing_Spout *Session , var = 'Distance.to.Spout')
v_slope_df <- test(pairs(v_slope), adjust = "bonferroni")
v_slope_df <- data.frame(v_slope_df)
print(v_slope_df)

v_slope <- emtrends(m1,~  Facing_Spout * Region | Session , var = 'Distance.to.Spout')
v_slope_df <- test(v_slope, adjust = "bonferroni")
v_slope_df <- data.frame(v_slope_df)
print(v_slope_df)

#print(as.data.frame(test(v_slope, adjust = "bonferroni")))
#print(test(pairs(v_slope), adjust = "bonferroni"))
v_slope <- as.data.frame(v_slope)

In [ ]:

col_order        = ['VS', 'TS']
x_col, y_col     = "Distance.to.Spout", "Z"
row_fac          = "Session"
col_fac          = "Region"
hue_col          = "Animal"

ROW_ORDER        = ["Reward", "Monster"]

X_LABEL = "Facing Spout"
Y_LABEL = "Slope (dZ / dDistance)"

FIG_HEIGHT   = 4.5
FIG_ASPECT   = 1.4

YLIM         = (-.02, 0.01)

POINT_ALPHA  = 0.25
POINT_SIZE   = 10
JITTER       = 0.10

MEAN_MS      = 2
MEAN_LW      = 2.0
MEAN_ALPHA   = 1.0
MEAN_CAPSIZE = 4

FACING_LEVELS = ["Front", "Side", "Behind"]

XTICK_LABELS = {
    "Front": "Front",
    "Side":  "Side",
    "Behind":"Behind",
}

REGION_COLORS = {
    "VS": "#1f77b4",
    "TS": "#E68600",
}

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 9.5,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 150,
})

def _slope_ols_1d(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2:
        return np.nan
    xm = x[m]; ym = y[m]
    if np.nanstd(xm) == 0:
        return np.nan
    X = np.vstack([xm, np.ones(xm.size)]).T 
    beta, *_ = np.linalg.lstsq(X, ym, rcond=None)
    return float(beta[0])

df_plot = df.copy()
df_plot = df_plot[df_plot[col_fac].isin(col_order)].copy()
df_plot["Facing_Spout"] = df_plot["Facing_Spout"].astype(str)

if row_fac in df_plot.columns:
    df_plot[row_fac] = pd.Categorical(df_plot[row_fac], categories=ROW_ORDER, ordered=True)

slopes = (
    df_plot
    .groupby([row_fac, col_fac, "Facing_Spout", hue_col], observed=True, sort=False)
    .apply(lambda g: _slope_ols_1d(g[x_col].to_numpy(), g[y_col].to_numpy()))
    .rename("slope").reset_index()
)

slopes[col_fac]          = pd.Categorical(slopes[col_fac],          categories=col_order,     ordered=True)
slopes["Facing_Spout"]   = pd.Categorical(slopes["Facing_Spout"],   categories=FACING_LEVELS, ordered=True)
slopes[row_fac]          = pd.Categorical(slopes[row_fac],          categories=ROW_ORDER,     ordered=True)

facing_pos = {a: i for i, a in enumerate(FACING_LEVELS)}
REGION_OFFSETS = {"VS": -0.15, "TS": +0.15}

trend = v_slope.copy()
for c in ["Session", "Region", "Facing_Spout"]:
    if c in trend.columns:
        trend[c] = trend[c].astype(str)

trend = trend[trend["Region"].isin(col_order)].copy()
trend["Region"]        = pd.Categorical(trend["Region"],        categories=col_order,     ordered=True)
trend["Facing_Spout"]  = pd.Categorical(trend["Facing_Spout"],  categories=FACING_LEVELS, ordered=True)
if "Session" in trend.columns:
    trend["Session"]   = pd.Categorical(trend["Session"],       categories=ROW_ORDER,     ordered=True)

n_rows = len(ROW_ORDER)
fig, axs = plt.subplots(
    nrows=n_rows, ncols=1, sharex=True, sharey=True,
    figsize=(FIG_HEIGHT*FIG_ASPECT, FIG_HEIGHT * n_rows),
    constrained_layout=False
)
axs = np.atleast_1d(axs)

rng = np.random.default_rng(1)
legend_handles = []
legend_labels  = []

for i, sess in enumerate(ROW_ORDER):
    ax = axs[i]

    sub_sess = slopes[slopes[row_fac] == sess]
    for region in col_order:
        sub = sub_sess[sub_sess[col_fac] == region]
        xs, ys = [], []
        for _, row in sub.iterrows():
            a = str(row["Facing_Spout"])
            x0 = facing_pos.get(a, np.nan)
            yv = row["slope"]
            if np.isfinite(x0) and np.isfinite(yv):
                xs.append(x0 + rng.uniform(-JITTER, +JITTER) + REGION_OFFSETS[region])
                ys.append(yv)
        if xs:
            sc = ax.scatter(xs, ys, s=POINT_SIZE, alpha=POINT_ALPHA,
                            color=REGION_COLORS[region], edgecolors="none",
                            label=region, zorder=2)
            if i == 0:
                legend_handles.append(sc)
                legend_labels.append(region)

    if "Session" in trend.columns:
        t_sess = trend[trend["Session"] == sess]
    else:
        t_sess = trend.copy()

    for a in FACING_LEVELS:
        x0 = facing_pos[a]
        for region in col_order:
            sub_a = t_sess[(t_sess["Facing_Spout"] == a) & (t_sess["Region"] == region)]
            if sub_a.empty:
                continue
            m  = float(pd.to_numeric(sub_a["Distance.to.Spout.trend"], errors="coerce").mean())
            se = float(pd.to_numeric(sub_a["SE"], errors="coerce").mean()) if "SE" in sub_a else np.nan
            if not np.isfinite(m):
                continue
            ax.errorbar([x0 + REGION_OFFSETS[region]], [m],
                        yerr=[[se]] if np.isfinite(se) else None,
                        fmt="o",
                        color=REGION_COLORS[region],
                        mfc="white", mec=REGION_COLORS[region], ecolor=REGION_COLORS[region],
                        ms=MEAN_MS, lw=MEAN_LW, capsize=MEAN_CAPSIZE,
                        alpha=MEAN_ALPHA, zorder=5.2, clip_on=False)

    ax.set_xlim(-0.5, len(FACING_LEVELS)-0.5)
    ax.set_ylim(*YLIM)
    ax.axhline(0, linestyle=(0, (4, 3)), linewidth=1.0, color="0.4", alpha=0.75, zorder=3.2)
    sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)

    ax.set_title(sess, loc="left", pad=2)

    ax.set_xticks([facing_pos[a] for a in FACING_LEVELS])
    ax.set_xticklabels([XTICK_LABELS.get(a, a) for a in FACING_LEVELS])
    ax.tick_params(axis='x', which='both', labelbottom=True)
fig.supylabel(Y_LABEL, x=0.02)
fig.supxlabel(X_LABEL, y=0.02)

fig.legend(handles=legend_handles, labels=legend_labels,
           loc="center left", bbox_to_anchor=(0.995, 0.5), frameon=False, title=None)

fig.tight_layout(rect=(0.03, 0.03, 0.98, 0.98))

plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Correlation_AngleDist.svg')

plt.show()
